# 05. CLAHE

**Objetivo:** comprender la diferencia entre ajustar el contraste globalmente y hacerlo por regiones, y explorar los parámetros principales de CLAHE.

In [ ]:
import numpy as np
import cv2

from filtrado_digital.io import cargar_imagen, a_grises, ruta_imagen_ejemplo
from filtrado_digital.visualizacion import mostrar_imagen, comparar

## 1. Contraste global y local

Un ajuste **global** usa la misma transformación para toda la imagen. Un ajuste **local** analiza regiones pequeñas y puede responder de forma diferente en cada zona.

**CLAHE** (*Contrast Limited Adaptive Histogram Equalization*) divide la imagen en pequeñas regiones o *tiles*, ajusta su contraste de manera local y limita la amplificación excesiva del histograma.

In [ ]:
foto = cargar_imagen(ruta_imagen_ejemplo())
original = a_grises(foto)

iluminacion = np.linspace(0.45, 1.15, original.shape[1])
no_uniforme = np.clip(original * iluminacion[None, :], 0, 255).astype(np.uint8)
mostrar_imagen(no_uniforme, "Iluminación no uniforme")

## 2. Ecualización global vs CLAHE

In [ ]:
global_eq = cv2.equalizeHist(no_uniforme)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
local_eq = clahe.apply(no_uniforme)
comparar([no_uniforme, global_eq, local_eq], ["Original", "Global", "CLAHE"])

## 3. `clipLimit`

`clipLimit` limita cuánto puede amplificarse localmente el histograma. Valores mayores suelen producir un efecto de contraste más fuerte, pero también pueden hacer más visible el ruido o la textura.

In [ ]:
resultados = []
titulos = []
for limite in [1.0, 2.0, 4.0]:
    clahe = cv2.createCLAHE(clipLimit=limite, tileGridSize=(8, 8))
    resultados.append(clahe.apply(no_uniforme))
    titulos.append(f"clipLimit={limite}")
comparar(resultados, titulos)

## 4. `tileGridSize`

`tileGridSize` indica en cuántas regiones se divide la imagen. Cambiar la rejilla modifica la escala espacial a la que se analiza el contraste.

In [ ]:
resultados = []
titulos = []
for rejilla in [(4, 4), (8, 8), (16, 16)]:
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=rejilla)
    resultados.append(clahe.apply(no_uniforme))
    titulos.append(f"grid={rejilla}")
comparar(resultados, titulos)

## Conclusiones

- CLAHE trabaja localmente, a diferencia de la ecualización global.
- `clipLimit` controla la amplificación local del contraste.
- `tileGridSize` determina la escala de las regiones analizadas.
- Un ajuste más intenso no siempre significa un resultado más útil.